# Notebook 3 (bonus): Key instance detection (KID)

Some multi-instance models don't just predict a property - they also learn *how much each conformer contributed*
to that prediction. In a multi-conformer model, this means the model can point at the specific conformer it thinks
is responsible for the molecule's activity - often called the **bioactive conformer**. Finding that conformer is
what we call **key instance detection (KID)**.

This notebook shows how to get those per-conformer weights out of a trained model, how to check whether they
actually point at the right conformer, and how to look at them visually.

**Before running this notebook**, install two extra packages that aren't part of core QSARmil (they're only needed
for this bonus notebook):

```bash
pip install huggingface_hub py3Dmol
```

In [ ]:
import pickle

import numpy as np
from rdkit import Chem

### 1. Load a dataset with known "correct answers"

To check whether a model's guesses about important conformers are actually right, we need a dataset where we
already know the true answer. Here we use a small, purpose-built benchmark: each molecule's bag contains up to 20
conformers, and a handful of them were deliberately designed to match specific pharmacophore patterns (simple 3D
"triggers" for activity). Those matching conformers are the true **key instances**; the rest are not.

The more pharmacophore patterns a conformer matches, the more "active" it's considered. Each molecule's target
value is simply the highest number of patterns matched by any of its conformers (from 1 to 7).

We download this dataset, plus two small helper scripts this notebook uses, from a Hugging Face dataset repository
(they're not part of the `qsarmil` package itself, since they're specific to this one demo).

In [ ]:
import importlib.util

from huggingface_hub import hf_hub_download

REPO_ID = "KagakuLab/QSARmil"


def load_module_from_hf(repo_id, filename, module_name, repo_type="dataset"):
    """Download a .py file from an HF repo and import it as a module, without touching sys.path."""
    path = hf_hub_download(repo_id, filename=filename, repo_type=repo_type)
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

In [ ]:
pkl_path = hf_hub_download(REPO_ID, filename="notebooks/train_conf.pkl", repo_type="dataset")
with open(pkl_path, "rb") as f:
    data_train = pickle.load(f)

pkl_path = hf_hub_download(REPO_ID, filename="notebooks/test_conf.pkl", repo_type="dataset")
with open(pkl_path, "rb") as f:
    data_test = pickle.load(f)

# each entry looks like: (an id, an RDKit Mol with several embedded conformers, the indices of the key
# conformers, the target value for the molecule)
len(data_train), len(data_test)

Just like in the other two notebooks, let's work with a small sample first, so the rest of this notebook runs
quickly. Comment this cell out once you're ready to use the full dataset.

In [ ]:
# uncomment to work with a small sample instead of the full dataset (much faster, good for a first try)
data_train = data_train[:20]
data_test = data_test[:10]
len(data_train), len(data_test)

In [ ]:
# molecules (RDKit Mol objects, each with multiple embedded conformers)
mols_train = [i[1] for i in data_train]
mols_test = [i[1] for i in data_test]


def split_into_conformers(mol):
    """Split an already-embedded multi-conformer Mol into a bag of single-conformer Mol objects."""
    bag = []
    for conf in mol.GetConformers():
        conf_mol = Chem.Mol(mol)
        conf_mol.RemoveAllConformers()
        conf_mol.AddConformer(conf, assignId=True)
        bag.append(conf_mol)
    return bag


# conformers - a plain list per molecule, no wrapper type (see Notebook 2 for more on this data shape)
confs_train = [split_into_conformers(i) for i in mols_train]
confs_test = [split_into_conformers(i) for i in mols_test]

# target property (highest number of matched pharmacophores in the bag, from 1 to 7)
y_train = [i[3] for i in data_train]
y_test = [i[3] for i in data_test]

# ground-truth key conformer indices for each molecule - this is what we'll check the model's guesses against
idx_train = [i[2] for i in data_train]
idx_test = [i[2] for i in data_test]

### 2. Computing descriptors

Same idea as in Notebook 2: each conformer needs to become a list of numbers before a model can learn from it. Any
3D descriptor works here in principle, but for KID specifically it helps to pick one that changes noticeably from
one conformer to another (otherwise there's nothing for the model to tell them apart by) and that's actually
related to the property you're predicting.

In [ ]:
from molfeat.calc import Pharmacophore3D
from qsarmil.descriptor.wrapper import DescriptorWrapper
from milearn.preprocessing import BagMinMaxScaler

desc_calc = DescriptorWrapper(Pharmacophore3D(factory="pmapper"), verbose=True)

In [ ]:
# DescriptorWrapper.run(...) returns a list of bags - see Notebook 2 for more detail
x_train = desc_calc.run(confs_train)
x_test = desc_calc.run(confs_test)


In [ ]:
scaler = BagMinMaxScaler()
scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

### 3. Training a model that can explain its own predictions

Not every multi-instance method can do KID - only the ones that use an internal weighting mechanism to combine
conformers. QSARmil has a few of these (see Notebook 2 for a full list of model families): the attention-based
networks (`AdditiveAttentionNetwork`, `SelfAttentionNetwork`, `HopfieldAttentionNetwork`) and
`DynamicPoolingNetwork`. All of them offer, in addition to the usual `model.predict(x)`:

- **`model.get_instance_weights(x)`** - returns, for each molecule, one weight per conformer in its bag. A higher
  weight means the model considered that conformer more important for its prediction.

We'll use `DynamicPoolingNetwork` here (see Notebook 2 for what each of its settings means).

In [ ]:
from milearn.network.module.hopt import DEFAULT_PARAM_GRID
from milearn.network.regressor import DynamicPoolingNetworkRegressor

model = DynamicPoolingNetworkRegressor(max_epochs=30)  # small max_epochs, just to keep this notebook fast
model.hopt(x_train_scaled, y_train, param_grid={**DEFAULT_PARAM_GRID, "max_epochs": 30}, verbose=True)
model.fit(x_train_scaled, y_train)

### 4. Checking whether the model's weights point at the right conformer

`kid_accuracy` (another small helper from the same Hugging Face repository) compares a model's predicted weights
against the true key conformers. It expects, for each molecule:

- `y_true`: a list of 0s and 1s, one per conformer, where `1` marks a true key conformer.
- `y_pred`: a list of the same length, with the model's predicted weight for each conformer.

It reports two numbers:
- **KID accuracy**: how often the single highest-weighted conformer (or the top `top_n`, if you ask for more than
  one) is actually a true key conformer.
- **Baseline accuracy**: what you'd get by picking conformers *at random* instead - useful for telling whether the
  model is actually better than guessing.

In [ ]:
from sklearn.metrics import r2_score

metrics = load_module_from_hf(REPO_ID, filename="notebooks/metrics.py", module_name="metrics")
kid_accuracy = metrics.kid_accuracy


def idx_to_binary(bags, key_indices):
    """Turn a list of key-conformer indices into a 0/1 label per conformer, one array per bag."""
    labels = []
    for bag, keys in zip(bags, key_indices):
        label = np.zeros(len(bag), dtype=int)
        label[keys] = 1
        labels.append(label)
    return labels


keys_test = idx_to_binary(confs_test, idx_test)

In [ ]:
y_pred = model.predict(x_test_scaled)
w_pred = model.get_instance_weights(x_test_scaled)
w_pred = [w.flatten() for w in w_pred]

top_n = 1

print(f"All molecules: {len(y_test)}")
print(f"Prediction accuracy (R2): {r2_score(y_test, y_pred):.2f}")

acc, exp = kid_accuracy(keys_test, w_pred, top_n=top_n)
print(f"KID accuracy: {acc:.2f}")
print(f"KID baseline accuracy (random guessing): {exp:.2f}")

### 5. Looking at the conformers directly

Numbers only tell you so much - it can help a lot to actually *look* at the conformers a model picked. The helper
below (from the same Hugging Face repository, needs `py3Dmol` as noted at the top of this notebook) draws a grid of
3D views: the true key conformer(s) highlighted in red, and the model's top-weighted guess(es) highlighted in
blue.

In [ ]:
visualization = load_module_from_hf(REPO_ID, filename="notebooks/visualization.py", module_name="visualization")
visualize_conformers_grid = visualization.visualize_conformers_grid

In [ ]:
N = 0  # pick a molecule index from the test set to inspect

print("predicted:", y_pred[N], "| true:", y_test[N])
visualize_conformers_grid(mols_test[N], w_pred[N], idx_test[N], top_n=3, sort_by_weight=True)

### What's next?

- `02_Professional_Pipeline_Customization.ipynb` explains every step used above (descriptors, model settings) in
  more detail.
- Try a different attention-based model (`AdditiveAttentionNetworkRegressor`, `SelfAttentionNetworkRegressor`,
  `HopfieldAttentionNetworkRegressor`) and compare their KID accuracy - not all of them will agree on which
  conformer matters most!